---
title: "2.5 Group summaries: the pivot table lesson"
---

::: {.lesson-tools}
[Open in Colab](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/lessons/02-data/05-group-summaries.ipynb){.btn .btn-primary}
[Download notebook](../../downloads/02-data/05-group-summaries.ipynb){.btn .btn-outline-secondary}
:::

**Run in your browser.** Colab needs a Google account to run and save work. If its GitHub link cannot open, download this notebook and use **File → Upload notebook** at [Colab](https://colab.research.google.com/). You can read the website without an account. A notebook is a document containing explanatory text and runnable code cells.

**Time:** about 40 minutes, including exercises · **Before you begin:** Complete [2.4 Missing values and data types](../02-data/04-missing-types.ipynb) first. Each notebook starts afresh: it does not remember variables from another lesson.

## You will learn

- Group observations and compute a mean with its contributing count.
- Distinguish equal-country and population-weighted summaries.
- Build a pivot-style summary while keeping the original table.

## Why it matters

The average for one region can be more informative than a long list of country rows—but only if you state what is averaged. Do countries receive equal weight, or do people? Are you comparing the same year? We begin with made-up numbers for two fictional country-years in each region, then summarize the real historical cross-section. Group summaries change the unit of observation.

## Worked example

### Split a table into groups

`.groupby("region")` organizes rows sharing a region label; selecting `"life_exp"` chooses a measurement and `.mean()` computes an average within each group. The small example has one year only. These are made-up numbers.

In [ ]:
import pandas as pd
small = pd.DataFrame({
    "country": ["Northland", "Eastria", "Southport"],
    "region": ["Northern group", "Northern group", "Southern group"],
    "year": [2007, 2007, 2007],
    "life_exp": [70.0, 80.0, 60.0],
    "population": [1_000_000, 3_000_000, 2_000_000],
})
small.groupby("region")["life_exp"].mean()

Northern group has a mean of 75.0 years and Southern group 60.0. Each country has equal weight in this mean. Grouping returns a result without collapsing or deleting the source table.

### Count the observations behind a mean

`.agg(...)` combines named summaries. A pair such as `("life_exp", "count")` identifies a column and an operation. `.size` counts rows; `.count` counts non-missing values in the selected column. `as_index=False` keeps region as an ordinary output column.

In [ ]:
small_summary = small.groupby("region", as_index=False).agg(
    mean_life_exp=("life_exp", "mean"),
    countries=("country", "size"),
    available_life_exp=("life_exp", "count"),
)
small_summary

The Northern group contains two country rows and the Southern group one. Here all life expectancies are available, so the two counts agree. With missing values they can differ. If a grouping label is missing, groupby drops it by default; use dropna=False when that unknown group needs to remain visible.

### Compare two weighting rules

A **weight** determines how much each observation contributes. Multiply life expectancy by population, sum those products, then divide by total population. This produces a population-weighted mean of country life expectancies.

In [ ]:
small["weighted_life_exp"] = small["life_exp"] * small["population"]
weighted_small = small.groupby("region").agg(
    numerator=("weighted_life_exp", "sum"),
    denominator=("population", "sum"),
)
weighted_small["population_weighted_life_exp"] = weighted_small["numerator"] / weighted_small["denominator"]
weighted_small[["population_weighted_life_exp"]]

The Northern group is now 77.5 years because Eastria has more people. This is not the same question as an equal-country mean. Even a population-weighted average of country life expectancies is not automatically an official regional life-table estimate; that requires the underlying mortality information.

### Prepare the real country panel

Run this prepared setup once in a fresh session. It uses the course CSV in a local checkout, or downloads its fixed, checksum-verified public source in Colab. A checksum is a fingerprint of a file's contents; a mismatch stops the download from being used. You do not need to retype the setup. It does not use your Google Drive or any private data.

In [ ]:
# Prepared setup: use the course CSV locally; fetch its pinned source in Colab.
from pathlib import Path
import hashlib
import io
from urllib.request import urlopen
import pandas as pd

data_file = Path("data/gapminder.csv")
if not data_file.exists():
    source = "https://raw.githubusercontent.com/jennybc/gapminder/391b5a40d3574c3b89d889dd7d72f7f3fdb4a70c/inst/extdata/gapminder.tsv"
    with urlopen(source, timeout=30) as response:
        raw = response.read()
    expected = "f74275b4a2f4a7c8e3a09a752f9d6e43d1759d78d5cbfdebaaa3e1a6efc68f78"
    if hashlib.sha256(raw).hexdigest() != expected:
        raise ValueError("The data download changed. Please report this to the course maintainer.")
    downloaded = pd.read_csv(io.BytesIO(raw), sep="\t", dtype=str)
    downloaded = downloaded.rename(columns={"continent": "region", "lifeExp": "life_exp", "pop": "population", "gdpPercap": "gdp_pc"})
    data_file.parent.mkdir(parents=True, exist_ok=True)
    downloaded.to_csv(data_file, index=False, lineterminator="\n")
print("Course data ready: data/gapminder.csv")

Look for **Course data ready**. Colab working files are temporary; rerun this cell after a runtime reset. The data cover 142 countries at five-year intervals from 1952 to 2007. Each row is one country in one year.

### Summarize the running example for one year

Load the real panel and choose 2007 before grouping. Keep the mean, median and contributing number of country observations. The **median** is the middle ordered value (or average of the two middle values).

In [ ]:
countries = pd.read_csv(data_file)
year_2007 = countries.loc[countries["year"] == 2007].copy()
region_summary = year_2007.groupby("region", as_index=False).agg(
    mean_life_exp=("life_exp", "mean"),
    median_life_exp=("life_exp", "median"),
    n_countries=("country", "size"),
    n_life_exp=("life_exp", "count"),
)
region_summary

Each output row is now a region in 2007. The five source groups contain different numbers of countries; Oceania contains only two in this extract. Differences in means can reflect composition, history and measurement. They do not identify an effect of region membership.

### Make a pivot-style display

`.pivot_table` summarizes rows into a grid: here years identify rows, regions identify columns and the cell is mean life expectancy. Specify the aggregation function rather than relying on a default.

In [ ]:
region_over_time = countries.pivot_table(index="year", columns="region", values="life_exp", aggfunc="mean")
region_over_time.head()

There are five regional means for each observed year. This averages countries equally within each group and year. It does not average all 1,704 rows into one cross-section or create measurements for unobserved years.

## Try it

Work in your own notebook copy. Add a new code cell for each task, try it yourself, then open the solution to compare. Run from the top if a name is missing.

### 1. Summarize income

Using the 2007 cross-section, create `income_summary` with the mean GDP per person and country count for each region.

<details><summary>Solution 1</summary>

Run this complete solution after the worked example.

In [ ]:
income_summary = year_2007.groupby("region", as_index=False).agg(
    mean_gdp_pc=("gdp_pc", "mean"), n_countries=("country", "size")
)
income_summary

The five rows are equal-country means. The mean is GDP per person averaged across countries, not total regional GDP divided by population.

</details>

### 2. Reconcile the sample

Sum `n_countries` in `region_summary`. Compare it with the number of rows in `year_2007`.

<details><summary>Solution 2</summary>

Run this complete solution after the worked example.

In [ ]:
summed_countries = region_summary["n_countries"].sum()
print(summed_countries)
print(len(year_2007))

Both counts are 142. This check helps catch dropped grouping labels or an accidental change in sample definition.

</details>

### 3. Check missing-value denominators

Create a practice copy of the small fictional table, blank Northland's life expectancy, and report both total rows and available life expectancies by group.

<details><summary>Solution 3</summary>

Run this complete solution after the worked example.

In [ ]:
practice = small.copy()
practice.loc[practice["country"] == "Northland", "life_exp"] = pd.NA
missing_summary = practice.groupby("region", as_index=False).agg(
    n_rows=("country", "size"), n_available=("life_exp", "count"), mean_life_exp=("life_exp", "mean")
)
missing_summary

Northern group still has two rows but only one available life expectancy. Its mean is now 80.0, computed from Eastria alone. A change in missingness can change a reported mean even without a change in the observed remaining value.

</details>

## When it goes wrong

The following examples deliberately trigger specific errors. The prepared `try`/`except` wrapper catches that named error and prints its message so the notebook can continue. It is for these demonstrations; an unexpected error elsewhere should still stop execution. You can read the message without memorizing the wrapper.

### The grouping field does not exist

This uses a column name from a different dataset.

In [ ]:
try:
    countries.groupby("continent")["life_exp"].mean()
except KeyError as error:
    print(f"{type(error).__name__}: {error}")

The course renamed the upstream continent field to region. Inspect the course data dictionary and columns.

### A mean is requested for text

Countries are labels, not numbers to average.

In [ ]:
try:
    year_2007["country"].mean()
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

Choose a numeric measurement and a meaningful summary. A text field may need a count or frequency table, not conversion into arbitrary numbers.

## Recap

- State the observation year, grouping rule and new unit of observation.
- Report counts and missingness alongside a mean.
- Equal-country and population-weighted summaries answer different questions.
- A pivot table summarizes; a plain pivot in lesson 2.7 only rearranges unique observations.

**Next:** [2.6 Combining datasets safely](../02-data/06-combining.ipynb).

::: {.callout-note title="Coming from Stata / R / Excel?"}
The operation resembles an Excel pivot table or Stata's collapse by a group. Assign the result to a new pandas name when you want to keep the original observations.
:::